# Exercise 06 — Working with Pre-trained Language Models

In the previous notebook you pretrained a 0.8 M parameter model for a few minutes on 1 MB of Shakespeare. The models in this notebook were made the same way — a Transformer, self-supervised training on raw text — but with a hundred times more parameters and many gigabytes of text. You will not train anything here: you **download** them with the Hugging Face [`transformers`](https://huggingface.co/docs/transformers/index) library and find out what pretraining has taught them.

There are three families of pre-trained Transformers. They differ in *which part* of the Transformer they keep and in their **pretraining objective**, and that decides what they are good at:

| family | model in this notebook | pretraining objective | each position sees | typically used for |
|---|---|---|---|---|
| encoder-only | **BERT** | *masked language modelling*: predict hidden tokens | the whole input, in both directions | understanding: classification, tagging, sentence and word vectors |
| decoder-only | **GPT-2** | *causal language modelling*: predict the next token | only the tokens to its left | generating text |
| encoder–decoder | **T5** | *span corruption*: write out the spans that were removed from the input | encoder: all of the input; decoder: its own output so far | text-to-text tasks: translation, summarisation, … |

## What you will do
1. **BERT** — fill in blanks, see why context on *both* sides matters, find out what else the model has picked up, and compare contextual word vectors.
2. **GPT-2** — compare it with your mini GPT, look at next-token distributions, measure perplexity, and sample text.
3. **T5** — span corruption, and one model for many tasks.
4. **Multiple-choice questions** — to check your understanding.

## How to work through it
Same as before: run the cells **in order**, fill in every `# TODO`, and make sure each **✅ Check** cell passes before moving on. Tasks that say *Your answer here* want a short written answer.

> **Heads up — this notebook downloads three models**, about 1.2 GB in total: BERT (~440 MB), GPT-2 (~550 MB) and T5-small (~240 MB). Each is downloaded the first time it is loaded and cached afterwards, so make sure you have a working internet connection. Everything runs on a CPU; nothing is trained.

In [1]:
import math

import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline,
)

torch.manual_seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")


def show(predictions):
    # Print a list of (token, probability) pairs
    for token, p in predictions:
        print(f"    {token!r:<16} {p:.3f}")

Using cpu device


## 1. BERT: an encoder that reads in both directions

[BERT](https://arxiv.org/abs/1810.04805) (Devlin et al., 2019) is a stack of 12 Transformer **encoder** layers: self-attention *without* a causal mask, so every token sees the whole sentence. It cannot be pretrained to predict the next token — it would simply look at it. Instead, 15% of the input tokens are hidden behind a `[MASK]` token, and the model has to reconstruct them: **masked language modelling** (MLM).

`AutoModelForMaskedLM` loads the encoder together with the MLM head, which turns every output vector into logits over the vocabulary — just like the `lm_head` of your GPT. (You may see a message that some weights were not used: those belong to BERT's second pretraining task, next sentence prediction, which we do not need.)

In [2]:
BERT_NAME = "google-bert/bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
bert = AutoModelForMaskedLM.from_pretrained(BERT_NAME).to(device).eval()

print(f"{sum(p.numel() for p in bert.parameters()):,} parameters, a vocabulary of {bert_tokenizer.vocab_size:,} tokens")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


109,514,298 parameters, a vocabulary of 30,522 tokens


BERT's tokenizer lowercases the text (this is the *uncased* model), splits it into **WordPiece** subword tokens, and adds two special tokens: `[CLS]` at the start (its output vector is used for classification) and `[SEP]` at the end (it separates two sentences in sentence-pair tasks).

In [3]:
inputs = bert_tokenizer("The capital of France is [MASK].", return_tensors="pt")
print(inputs["input_ids"])
print(bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

tensor([[ 101, 1996, 3007, 1997, 2605, 2003,  103, 1012,  102]])
['[CLS]', 'the', 'capital', 'of', 'france', 'is', '[MASK]', '.', '[SEP]']


### Filling in the blank

**1.1 Complete `mask_distribution`**, which returns BERT's probability distribution over the vocabulary at the `[MASK]` position. `fill_mask` uses it to list the most likely tokens.

Hints:
- `bert(**inputs)` returns an object with the attribute `.logits`, of shape `(1, seq_len, vocab_size)`.
- `(ids == bert_tokenizer.mask_token_id).nonzero()` finds the position of the mask.

In [4]:
def mask_distribution(text):
    # BERT's probability for every token of the vocabulary at the (single) [MASK] in text, shape (vocab_size,)
    inputs = bert_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = bert(**inputs).logits  # TODO: run BERT, (1, seq_len, vocab_size)
    mask_index = (inputs["input_ids"][0] == bert_tokenizer.mask_token_id).nonzero().item()  # TODO: the position of the [MASK] token, as a Python int
    return logits[0, mask_index].softmax(dim=-1)        # TODO: softmax over the vocabulary at that position


def fill_mask(text, k=5):
    # The k most likely tokens for the [MASK], as a list of (token, probability)
    top = torch.topk(mask_distribution(text), k)
    return list(zip(bert_tokenizer.convert_ids_to_tokens(top.indices.tolist()), top.values.tolist()))


show(fill_mask("The capital of France is [MASK]."))

    'paris'          0.417
    'lille'          0.071
    'lyon'           0.063
    'marseille'      0.044
    'tours'          0.030
Device: cpu
BERT device: cpu


In [5]:
# ✅ Check your mask_distribution
probs = mask_distribution("The capital of France is [MASK].")
assert probs.shape == (bert_tokenizer.vocab_size,), f"expected one probability per token of the vocabulary, got shape {tuple(probs.shape)}"
assert torch.isclose(probs.sum(), torch.tensor(1.0, device=probs.device), atol=1e-4), "the probabilities should sum to 1"
assert fill_mask("The capital of France is [MASK].")[0][0] == "paris", "BERT should know the capital of France"
print("Looks good ✅")

Looks good ✅


The Hugging Face **pipelines** wrap tokenization, the model and the post-processing into a single call. Your `fill_mask` is what the `fill-mask` pipeline does under the hood:

In [9]:
unmasker = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer, device=device)
for prediction in unmasker("The capital of France is [MASK]."):
    print(f"    {prediction['token_str']!r:<16} {prediction['score']:.3f}")

    'paris'          0.417
    'lille'          0.071
    'lyon'           0.063
    'marseille'      0.044
    'tours'          0.030


### Context on both sides

**1.2 Run the cell below.** In each pair, the words *before* the mask are identical.

In [10]:
for text in [
    "I went to the [MASK] to buy some bread.",
    "I went to the [MASK] to see a doctor.",
    "The [MASK] barked at the postman all night.",
    "The [MASK] purred on my lap all night.",
]:
    print(text)
    show(fill_mask(text))

I went to the [MASK] to buy some bread.
    'market'         0.780
    'store'          0.056
    'bakery'         0.047
    'kitchen'        0.023
    'markets'        0.015
I went to the [MASK] to see a doctor.
    'hospital'       0.810
    'clinic'         0.033
    'bathroom'       0.027
    'office'         0.012
    'doctor'         0.009
The [MASK] barked at the postman all night.
    'dog'            0.345
    'dogs'           0.073
    'man'            0.057
    'guard'          0.023
    'cat'            0.021
The [MASK] purred on my lap all night.
    'cat'            0.228
    'dog'            0.226
    'baby'           0.096
    'kitten'         0.058
    'girl'           0.034


**1.3 What does this show?** What would a left-to-right model like your mini GPT (or GPT-2) predict at the position of the mask, and why can BERT do better? Conversely: why is BERT not a good model for *generating* text?

---

*Your answer here:*

---

### What else did BERT learn?

BERT learned everything it knows from its pretraining text (English Wikipedia and the BooksCorpus collection of books). That includes facts about the world — and also the patterns of how people are written about.

**1.4 Run the cell below**, which compares the probabilities of *he* and *she* at the mask. Add a few sentences of your own.

In [12]:
he_id, she_id = bert_tokenizer.convert_tokens_to_ids(["he", "she"])

for job in ["nurse", "engineer", "secretary", "doctor", "teacher", "pilot", "thief", "idiot", "cat", "dog"]:
    probs = mask_distribution(f"The {job} said that [MASK] would be late.")
    print(f"{job:<12} he: {probs[he_id]:.3f}   she: {probs[she_id]:.3f}")

nurse        he: 0.175   she: 0.483
engineer     he: 0.409   she: 0.051
secretary    he: 0.521   she: 0.177
doctor       he: 0.337   she: 0.298
teacher      he: 0.362   she: 0.303
pilot        he: 0.465   she: 0.064
thief        he: 0.697   she: 0.135
idiot        he: 0.458   she: 0.171
cat          he: 0.293   she: 0.292
dog          he: 0.442   she: 0.170


**1.5 Describe the pattern.** Where does it come from — is it a bug in the model? Why does it matter if you fine-tune BERT for a real application, for example to rank job applications?

---

*Your answer here:*

---

### Contextual word vectors

In week 3 you worked with **static** word embeddings: word2vec gives the word *bank* one vector, whatever it means in a sentence. BERT's output vectors are **contextual**: every occurrence of a token gets its own vector, computed from the whole sentence.

**1.6 Complete `word_vector`**, which returns BERT's last-layer vector for a word in a sentence.

Hints:
- With `output_hidden_states=True` the model also returns `.hidden_states`: a tuple with the output of the embedding layer and of each of the 12 layers, each of shape `(1, seq_len, 768)`. The last element is the output of the last layer.
- The word must be a single token in BERT's vocabulary; find its position with `bert_tokenizer.convert_tokens_to_ids(word)`.

In [25]:
def word_vector(sentence, word):
    # BERT's last-layer vector of `word` (a single token) in `sentence`, shape (768,)
    inputs = bert_tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = bert(**inputs, output_hidden_states=True).hidden_states[-1][0]  # TODO: the last-layer hidden states of the sentence, (seq_len, 768)
    position = (inputs["input_ids"][0] == bert_tokenizer.convert_tokens_to_ids(word)).nonzero()[0].item()    # TODO: the position of the word's token in inputs["input_ids"][0] (the first, if it occurs twice)
    return hidden[position]

In [26]:
# ✅ Check your word_vector
v1 = word_vector("He sat on the bank of the river.", "bank")
v2 = word_vector("She works at the bank.", "bank")
assert v1.shape == (768,), f"expected a vector of size 768, got shape {tuple(v1.shape)}"
assert not torch.allclose(v1, v2), "the same word in two sentences should get two different vectors"
assert torch.allclose(v1, word_vector("He sat on the bank of the river.", "bank")), "the same sentence should give the same vector"
print("Looks good ✅")

Looks good ✅


**1.7 Run the cell below.** It compares the vectors of *bank* in four sentences with the cosine similarity.

In [28]:
sentences = [
    "He sat on the bank of the river and watched the water.",
    "The fishermen pulled their boat onto the muddy bank.",
    "She deposited the money at the bank this morning.",
    "The bank raised its interest rates again.",
]
vectors = torch.stack([word_vector(s, "bank") for s in sentences])
similarity = F.cosine_similarity(vectors[:, None], vectors[None, :], dim=-1)

for i, s in enumerate(sentences):
    print(f"{i}: {s}")
print("\n     " + "".join(f"{j:>7}" for j in range(len(sentences))))
for i in range(len(sentences)):
    print(f"{i:>3}  " + "".join(f"{similarity[i, j]:>7.2f}" for j in range(len(sentences))))

0: He sat on the bank of the river and watched the water.
1: The fishermen pulled their boat onto the muddy bank.
2: She deposited the money at the bank this morning.
3: The bank raised its interest rates again.

           0      1      2      3
  0     1.00   0.80   0.47   0.42
  1     0.80   1.00   0.47   0.48
  2     0.47   0.47   1.00   0.76
  3     0.42   0.48   0.76   1.00


**1.8 Describe the similarity matrix.** What would the matrix look like with word2vec? Why are the vectors of the same *sense* not identical, and why are vectors of *different* senses still fairly similar?

---

*Your answer here:*

---

## 2. GPT-2: your mini GPT, grown up

[GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) (Radford et al., 2019) is a decoder-only Transformer trained to predict the next token on 40 GB of web text. We use the smallest version, GPT-2 *small*. `AutoModelForCausalLM` loads it with its language-modelling head.

In [29]:
GPT2_NAME = "openai-community/gpt2"

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_NAME)
gpt2 = AutoModelForCausalLM.from_pretrained(GPT2_NAME).to(device).eval()

print(gpt2)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


Compare the printout with your `GPT` class: `wte` and `wpe` are the token and position embeddings, `h` is the list of blocks, each with `ln_1`, `attn`, `ln_2` and `mlp`, followed by `ln_f` and the `lm_head`. `c_attn` is your `qkv` layer — note its output size. (`Conv1D` is simply a linear layer that stores its weight matrix transposed, a leftover from the original TensorFlow code.)

**2.1 Count the parameters of GPT-2, and check whether its head is tied to the token embedding.**

Hint: `gpt2.get_input_embeddings()` and `gpt2.get_output_embeddings()` return the two layers. Two tensors share the same memory if their `.data_ptr()` is the same.

In [37]:
n_params = sum(p.numel() for p in gpt2.parameters())  # TODO: the number of parameters of GPT-2
tied = gpt2.transformer.wte.weight.data_ptr() == gpt2.lm_head.weight.data_ptr()      # TODO: True if the head and the token embedding share their weight matrix

print(f"{n_params:,} parameters, tied: {tied}")

124,439,808 parameters, tied: True


In [38]:
# ✅ Check your numbers
assert 120e6 < n_params < 130e6, "GPT-2 small has about 124 million parameters"
assert tied is True, "GPT-2 ties its output layer to the token embedding"
print("Looks good ✅")

Looks good ✅


### Subword tokens

Instead of characters, GPT-2 reads **byte-level BPE** tokens: frequent words are a single token, rare words are split into pieces. `Ġ` marks a token that starts with a space. You will learn how such a vocabulary is built later in the course.

In [39]:
text = "Shakespeare wrote: To be, or not to be. Pretraining is unbelievably expensive!"
tokens = gpt2_tokenizer.tokenize(text)
print(f"{len(text)} characters -> {len(tokens)} tokens")
print(tokens)

78 characters -> 18 tokens
['Sh', 'akespeare', 'Ġwrote', ':', 'ĠTo', 'Ġbe', ',', 'Ġor', 'Ġnot', 'Ġto', 'Ġbe', '.', 'ĠPret', 'raining', 'Ġis', 'Ġunbelievably', 'Ġexpensive', '!']


In [41]:
mini = { "layers": 4, "embedding": 128, "context": 64, "vocab": 65, }
gpt2_specs = {
    "layers": gpt2.config.n_layer,
    "embedding": gpt2.config.n_embd,
    "context": gpt2.config.n_positions,
    "vocab": gpt2.config.vocab_size,
}
print(f"{'Property':<18} {'Mini GPT':>12} {'GPT-2':>12}")
for key in mini:
    print(f"{key:<18} {mini[key]:>12,} {gpt2_specs[key]:>12,}")
# Compare how much text fits into the context
sample = "Shakespeare wrote: To be, or not to be. Pretraining is unbelievably expensive!"
n_tokens = len(gpt2_tokenizer.encode(sample))

print(f"\nMini GPT context: {mini['context']} characters")
print(f"GPT-2 context: {gpt2_specs['context']} BPE tokens")
print(f"GPT-2 approximate capacity for this sample: "
      f"{gpt2_specs['context'] * len(sample) / n_tokens:.0f} characters")

Property               Mini GPT        GPT-2
layers                        4           12
embedding                   128          768
context                      64        1,024
vocab                        65       50,257

Mini GPT context: 64 characters
GPT-2 context: 1024 BPE tokens
GPT-2 approximate capacity for this sample: 4437 characters


**2.2 Your mini GPT and GPT-2 small.** Using the printout and the cells above, compare the two models: number of layers, embedding dimension, context length, vocabulary, and how much text fits into the context. What is the same in the two architectures?

---

*Your answer here:*

---

### The next token

**2.3 Complete `next_token_distribution`**, which returns GPT-2's most likely next tokens after a prompt.

Hint: the model returns logits for every position of the prompt. Which position predicts the token *after* the prompt?

In [51]:
def next_token_distribution(prompt, k=10):
    # The k most likely tokens after the prompt, as a list of (token, probability)
    inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = gpt2(**inputs).logits  # TODO: run GPT-2, (1, seq_len, vocab_size)
    probs = logits[0, -1].softmax(dim=-1)       # TODO: the probability distribution over the token after the prompt, (vocab_size,)
    top = torch.topk(probs, k)
    return [(gpt2_tokenizer.decode([i]), p) for i, p in zip(top.indices.tolist(), top.values.tolist())]


show(next_token_distribution("The capital of France is"))

    ' the'           0.085
    ' now'           0.048
    ' a'             0.046
    ' France'        0.032
    ' Paris'         0.032
    ' in'            0.027
    ' also'          0.026
    ' not'           0.024
    ' home'          0.023
    ' still'         0.016


In [52]:
# ✅ Check your next_token_distribution
top = next_token_distribution("The capital of France is")
assert len(top) == 10 and all(a[1] >= b[1] for a, b in zip(top, top[1:])), "expected the 10 most likely tokens, sorted by probability"
assert any(token.strip() == "Paris" for token, _ in top), "' Paris' should be among the 10 most likely next tokens — are you using the last position?"
print("Looks good ✅")

Looks good ✅


Unlike your Shakespeare model, GPT-2 knows that Paris is the capital of France — but it is not necessarily its first choice. Try a few prompts of your own:

In [53]:
for prompt in [
    "The Eiffel Tower is located in the city of",
    "Aarhus is the second largest city in",
    "Q: What is 12 plus 7?\nA:",
]:
    print(repr(prompt))
    show(next_token_distribution(prompt, k=5))

'The Eiffel Tower is located in the city of'
    ' Paris'         0.064
    ' London'        0.046
    ' Amsterdam'     0.034
    ' New'           0.032
    ' Berlin'        0.026
'Aarhus is the second largest city in'
    ' the'           0.314
    ' India'         0.138
    ' Denmark'       0.122
    ' northern'      0.028
    ' North'         0.019
'Q: What is 12 plus 7?\nA:'
    ' 12'            0.144
    ' It'            0.075
    ' The'           0.068
    ' I'             0.042
    ' This'          0.031


### Perplexity

A language model assigns a probability to every text. The **perplexity** (week 3) turns it into a number: the exponential of the average negative log-probability of the tokens. The lower, the less surprised the model is.

If you pass `labels` to the model, it computes the loss itself — the same next-token cross-entropy you computed in your `GPT.forward`. You pass the input ids as the labels; the model shifts them by one position internally.

**2.4 Complete `perplexity`.**

In [64]:
def perplexity(text):
    inputs = gpt2_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        loss = gpt2(**inputs, labels=inputs["input_ids"]).loss  # TODO: the language-modelling loss of the text (pass labels=...)
    return math.exp(loss.item())


print(perplexity("The cat sat on the mat."))

90.24211658357754


In [65]:
# ✅ Check your perplexity against a computation by hand
text = "The cat sat on the mat."
inputs = gpt2_tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    logits = gpt2(**inputs).logits
by_hand = math.exp(F.cross_entropy(logits[0, :-1], inputs["input_ids"][0, 1:]).item())  # position t predicts token t + 1
assert abs(perplexity(text) - by_hand) < 1e-3 * by_hand, f"expected {by_hand:.2f}, got {perplexity(text):.2f}"
print("Looks good ✅")

Looks good ✅


**2.5 Run the cell below**, and add texts of your own.

In [77]:
for text in [
    "The cat sat on the mat.",
    "The mat sat on the cat.",
    "mat the on sat cat The.",
    "Colorless green ideas sleep furiously.",
    "To be, or not to be, that is the question.",
    "Katten sad på måtten.",
    "for i in range(10):\n    print(i)",
    "for i in range(10);\n    print(i)",
    "Der var en gang for længe siden. Sådan startede historien.",
    "blue snow",
    "yellow snow",
    "He is an idiot.",
    "She is an idiot.",
    "He is smart",
    "She is smart",
    "He is very intelligent.",
    "She is very intelligent.",
]:
    print(f"{perplexity(text):>10.1f}   {text!r}")

      90.2   'The cat sat on the mat.'
     383.9   'The mat sat on the cat.'
    6037.5   'mat the on sat cat The.'
    6413.3   'Colorless green ideas sleep furiously.'
      18.4   'To be, or not to be, that is the question.'
     828.3   'Katten sad på måtten.'
      50.1   'for i in range(10):\n    print(i)'
      61.2   'for i in range(10);\n    print(i)'
     686.1   'Der var en gang for længe siden. Sådan startede historien.'
   20693.3   'blue snow'
   13895.8   'yellow snow'
      35.2   'He is an idiot.'
      44.2   'She is an idiot.'
     246.4   'He is smart'
     186.7   'She is smart'
      36.0   'He is very intelligent.'
      29.5   'She is very intelligent.'


**2.6 Explain the differences.** Which texts surprise GPT-2 the most, and why? Can you compare these numbers with the validation loss of your Shakespeare model from notebook 0?

---

*Your answer here:*

"Blue snow" - suspect that it is because that combination is very unlikely, and then a short sentence increases the likelihood because it is surprised on every position.

For some of the longer texts like "Colorless green ideas sleep furiously.", it is still very "surprised", but some of the combinations are
still less surprising for some reason.

No. Especially because the tokenization is completely different which makes it unfair.

---

### Generating text

You implemented sampling with temperature and top-k yourself. Hugging Face models have a `generate` method that does this (and much more). Its most important arguments:

- `max_new_tokens`: how many tokens to generate,
- `do_sample`: `False` for greedy decoding, `True` for sampling,
- `temperature` and `top_k`: as in your `generate`,
- `top_p` (*nucleus sampling*): keep the smallest set of tokens whose probabilities add up to `top_p`, so the number of candidates adapts to how certain the model is.

GPT-2 has no padding token, so we pass `pad_token_id=gpt2_tokenizer.eos_token_id` to avoid a warning.

**2.7 Generate a continuation of the prompt with greedy decoding and with nucleus sampling.**

In [80]:
prompt = "In a shocking finding, scientists discovered a herd of unicorns living in a remote valley in the Andes."
inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)

torch.manual_seed(0)
greedy = gpt2.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False,
    pad_token_id=gpt2_tokenizer.eos_token_id,
)   # TODO: gpt2.generate with greedy decoding and 80 new tokens
sampled = gpt2.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    pad_token_id=gpt2_tokenizer.eos_token_id,
)  # TODO: gpt2.generate with sampling, temperature=0.8, top_p=0.9 and 80 new tokens

print("===== greedy =====")
print(gpt2_tokenizer.decode(greedy[0], skip_special_tokens=True))
print("\n===== sampled =====")
print(gpt2_tokenizer.decode(sampled[0], skip_special_tokens=True))

===== greedy =====
In a shocking finding, scientists discovered a herd of unicorns living in a remote valley in the Andes.

The researchers found that the animals were able to survive in a climate that was warmer than the average for the region.

"The animals were able to survive in a climate that was warmer than the average for the region," said lead author Dr. David H. Hirsch, a professor of ecology and evolutionary biology at the University of California, Berkeley.

The researchers found that the

===== sampled =====
In a shocking finding, scientists discovered a herd of unicorns living in a remote valley in the Andes. The study, published in the Journal of Zoology, also found that the animals had a unique relationship with the land and that they grew up in a similar situation.

Researchers with the National Science Foundation, a division of the National Science Foundation, and the National Institute of Allergy and Infectious Diseases (NIH), a division of the National Institutes of 

In [81]:
# ✅ Check your generations
prompt_length = inputs["input_ids"].size(1)
assert prompt_length < greedy.shape[1] <= prompt_length + 80 and prompt_length < sampled.shape[1] <= prompt_length + 80, "generate up to 80 new tokens after the prompt"
again = gpt2.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=gpt2_tokenizer.eos_token_id)
assert torch.equal(greedy, again), "greedy decoding should be deterministic — did you set do_sample=False?"
print("Looks good ✅")

Looks good ✅


Now try the model as an assistant:

In [82]:
torch.manual_seed(0)
prompt = "Question: What is the capital of Denmark?\nAnswer:"
inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)
output = gpt2.generate(**inputs, max_new_tokens=40, do_sample=True, top_p=0.9, pad_token_id=gpt2_tokenizer.eos_token_id)
print(gpt2_tokenizer.decode(output[0], skip_special_tokens=True))

Question: What is the capital of Denmark?
Answer: The capital of Denmark is Copenhagen and is the home of the Danish Parliament. Danish voters vote by telephone and on polling day. In 2011, a significant majority of Danish voters decided against electing a new leader


**2.8 Compare the outputs.** What typically goes wrong with greedy decoding? How does the sampled text compare? And why does GPT-2 not behave like a chat assistant, even though it was trained on far more text than your Shakespeare model?

---

*Your answer here:*

---

## 3. T5: one model for every task

[T5](https://arxiv.org/abs/1910.10683) (Raffel et al., 2020), the *Text-to-Text Transfer Transformer*, is an **encoder–decoder** — the architecture you built last week. It was pretrained with **span corruption**: random spans of the input are replaced by *sentinel* tokens `<extra_id_0>`, `<extra_id_1>`, …, and the decoder has to write out the missing spans, each one after its sentinel:

```
original   Thank you for inviting me to your party last week.
input      Thank you <extra_id_0> me to your party <extra_id_1> week.
target     <extra_id_0> for inviting <extra_id_1> last <extra_id_2>
```

We use T5-*small* (60 M parameters).

In [83]:
T5_NAME = "google-t5/t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(T5_NAME)
t5 = AutoModelForSeq2SeqLM.from_pretrained(T5_NAME).to(device).eval()

print(f"{sum(p.numel() for p in t5.parameters()):,} parameters")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

60,506,624 parameters


**3.1 Complete `t5_generate`**, which runs T5 on a text and returns its output as a string.

Hints:
- `t5.generate(**inputs, max_new_tokens=...)` returns a tensor of token ids of shape `(1, output_length)`. By default it uses greedy decoding.
- `t5_tokenizer.decode(ids, skip_special_tokens=...)`: the sentinels are special tokens, so to see them you have to keep the special tokens.

In [94]:
def t5_generate(text, max_new_tokens=40, skip_special_tokens=True):
    inputs = t5_tokenizer(text, return_tensors="pt").to(device)
    outputs = t5.generate(**inputs, max_new_tokens=max_new_tokens)  # TODO: generate the output
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=skip_special_tokens)     # TODO: decode the first (and only) output sequence


print(t5_generate("Thank you <extra_id_0> me to your party <extra_id_1> week.", skip_special_tokens=False))

<pad> Thank you for inviting<extra_id_1> this<extra_id_2> for the wonderful party.</s>


In [95]:
# ✅ Check your t5_generate
corrupted = t5_generate("Thank you <extra_id_0> me to your party <extra_id_1> week.", skip_special_tokens=False)
assert isinstance(corrupted, str), "t5_generate should return a string"
assert "<extra_id_0>" in corrupted, "the output of span corruption should contain the sentinel <extra_id_0> — keep the special tokens"
translation = t5_generate("translate English to German: The house is wonderful.")
print(translation)
assert "Haus" in translation and "<" not in translation, "expected a German translation without special tokens"
print("Looks good ✅")

AssertionError: the output of span corruption should contain the sentinel <extra_id_0> — keep the special tokens

### Text to text

The idea of T5 is that **every task is a text-to-text task**: the input is a text that starts with a *task prefix*, the output is a text. Translation outputs the translation; classification outputs the name of the class; even a regression task outputs a number, written as text. The released T5 checkpoints were trained on a mixture of the span-corruption objective and such supervised tasks, so they understand a few prefixes out of the box.

**3.2 Run the cell below.** Then change the inputs: other sentences, other prefixes, a typo in a prefix, or a language that T5 was not trained on (it knows German, French and Romanian).

In [96]:
article = (
    "summarize: The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889. "
    "Locally nicknamed 'La dame de fer', it was constructed as the centerpiece of the 1889 World's Fair, "
    "and to crown the centennial anniversary of the French Revolution. Although initially criticised by some of "
    "France's leading artists and intellectuals for its design, it has since become a global cultural icon of France "
    "and one of the most recognisable structures in the world."
)

for text in [
    "translate English to German: The house is wonderful.",
    "translate English to French: Pretraining a language model takes a lot of compute.",
    "translate English to Danish: The house is wonderful.",
    article,
    "cola sentence: The books is on the table.",
    "cola sentence: The books are on the table.",
    "sst2 sentence: This movie was a complete waste of time.",
    "stsb sentence1: A man is playing a guitar. sentence2: A person plays an instrument.",
]:
    print(f"{text[:80]!r}\n    -> {t5_generate(text, max_new_tokens=60)!r}\n")

'translate English to German: The house is wonderful.'
    -> 'Das Haus ist wunderbar.'

'translate English to French: Pretraining a language model takes a lot of compute'
    -> "La formation préscolaire d'un modèle linguistique nécessite beaucoup de calcul."

'translate English to Danish: The house is wonderful.'
    -> 'Das Haus ist wunderbar.'

'summarize: The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars'
    -> 'the tower was built by the engineer Gustave Eiffel from 1887 to 1889. it was named after the engineer who designed and built the tower.'

'cola sentence: The books is on the table.'
    -> 'acceptable'

'cola sentence: The books are on the table.'
    -> 'acceptable'

'sst2 sentence: This movie was a complete waste of time.'
    -> 'negative'

'stsb sentence1: A man is playing a guitar. sentence2: A person plays an instrume'
    -> '3.6'



**3.3 How does T5 solve classification and regression tasks?** What happened with the Danish prefix (or with your typos), and what does that tell you about what the prefixes are?

---

*Your answer here:*

---

**3.4 Which family would you choose?** For each task, choose an encoder-only, a decoder-only or an encoder–decoder model, and explain briefly:
1. classify customer reviews as positive or negative,
2. a system that writes short stories from a first sentence,
3. translate product descriptions from English to Danish,
4. find the 10 most similar documents in a large collection.

---

*Your answer here:*

1. Encoder-only (BERT): The whole input is available,
2. Decoder-only (GPT): This is text generation like GPT
3. Encoder-decoder (T5, BART, or dedicated): The encoder can read the whole input, then a decoder can generate the output sequence using cross attention.
4. Encoder-decoder: Just "encode" every document into a vector, then take the 10 most similar ones. There is nothing to decode.

---

## 4. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 4.1. BART's hybrid design

[BART](https://arxiv.org/abs/1910.13461) combines which two ideas?

A. Bidirectional encoder + autoregressive decoder<br>
B. Masked LM + next sentence prediction<br>
C. LSTM + CNN<br>
D. Word2Vec + Transformers<br>

**Answer:** 
A. (The others are wrong).

---

### 4.2. BART vs BERT

Which of the following is TRUE about BART compared to BERT?

A. BART cannot handle generative tasks<br>
B. BART performs worse on classification tasks<br>
C. BART can do both understanding and generation<br>
D. BART uses only unidirectional encoding<br>

**Answer:**
C.

---

### 4.3. BART architecture

What architecture does BART use?

A. Encoder-only<br>
B. Decoder-only<br>
C. Encoder–decoder with cross-attention<br>
D. RNN-based encoder<br>

**Answer:**
C.

---

### 4.4. BART's advantage

Why does BART outperform BERT on some tasks?

A. It has fewer parameters<br>
B. It was trained on significantly more data<br>
C. It does not use positional embeddings<br>
D. It uses static word embeddings<br>

**Answer:**
B.

---

### 4.5. BART corruption strategies

Which of the following corruption strategies can BART use during pretraining?

A. Masking tokens randomly<br>
B. Permuting sentences<br>
C. Replacing tokens with random ones<br>
D. All of the above<br>

**Answer:**
D.

---

### 4.6. T5's name

What does T5 stand for?

A. Text-to-Text Transfer Transformer<br>
B. Transformer for 5 tasks<br>
C. Transferable Transformer Training Technique<br>
D. Text Transformer for Translation<br>

**Answer:**
A.

---

### 4.7. T5's philosophy

What is the key design philosophy of T5?

A. Every NLP task can be framed as a text-to-text problem<br>
B. Using unidirectional LSTMs<br>
C. Using static embeddings<br>
D. Relying only on classification tasks<br>

**Answer:**
A.

---

### 4.8. T5's pretraining objective

Which pretraining objective did T5 use?

A. Masked language modeling (like BERT)<br>
B. Next sentence prediction<br>
C. Denoising span corruption (span masking)<br>
D. Causal LM (like GPT)<br>

**Answer:**
C.

---

### 4.9. T5's architecture

Which type of Transformer architecture does T5 employ?

A. Encoder-only<br>
B. Decoder-only<br>
C. Encoder–decoder<br>
D. Hybrid LSTM-Transformer<br>

**Answer:**
C.

---

### 4.10. T5's limitations

Which of the following is NOT a task T5 can handle natively in its framework?

A. Machine translation<br>
B. Summarization<br>
C. Image classification<br>
D. Question answering<br>

**Answer:**
B. X\
Correct is C. T5 is text-to-text